<a href="https://colab.research.google.com/github/Carlos-V-V/Gen-AI-for-CCM-pipeline-prototypes/blob/main/v3_Interaction_Networks_pipeline_No_OT_HJB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DESCRIPTION OF THIS VERSION**

This copy of the pipeline notebook defines the Interaction Networks that will be used to model the learned interactions in our pipeline, defines the ODE system, the cost functional J, and then does the training loop, which trains our NNs with the optimization problem defined in the paper draft / slides. After training, then we can visualize the predicted cell trajectories in animation form, plot the behavior of the training loss over training epochs, save all the outputs as CSV files, and save the trained NN models.

**NEXT STEPS** :

- Adapt this to include the Rayleighian regularizer + enforce the discrete-time HJB optimality condition.

**ASSUMED MODEL**

This run of the pipeline assumes the system of equations:

dx_i/dt = Morse interactions

dp_i/dt = 0

(Active migration term has to be learned by the Interaction Network)

In [ ]:
!pip install torchdiffeq
!pip install geomloss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchdiffeq import odeint  # from the `torchdiffeq' package
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cpu


We define the known kernels to use in this iteration. We define them in this way to be able to integrate the kernel parameters into the optimizing pipeline, so that backpropagation flows through both the NN parameters (\phi) and the kernel parameters (\theta). That is, the following is a trainable kernel model.  

We define our parameters module:

In [ ]:
class GlobalKernelParams(nn.Module):
    def __init__(self, init_A = 0.1 , init_a=1.0, init_R=0.1, init_r=1.0):
        super().__init__()
        # store unconstrained "raw" parameters
        # softplus will keep them positive
        self.raw_A = nn.Parameter(torch.tensor(float(init_A)).log().exp().log())
        self.raw_a = nn.Parameter(torch.tensor(float(init_a)).log().exp().log())  # okay-ish init
        self.raw_R = nn.Parameter(torch.tensor(float(init_R)).log().exp().log())
        self.raw_r = nn.Parameter(torch.tensor(float(init_r)).log().exp().log())

    def forward(self):
        A   = F.softplus(self.raw_A)       # > 0
        a   = F.softplus(self.raw_a)       # > 0
        R   = F.softplus(self.raw_R)       # > 0
        r   = F.softplus(self.raw_r)     # > 0
        return {"A": A, "a": a, "R": R, "r": r}

Now the NN's modeling the known kernels:

In [ ]:
class PosKernels(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, positions, params):
    # params: dict with tensors (scalars)
        A = params["A"]
        a = params["a"]
        R = params["R"]
        r = params["r"]

        '''
        Previous version of this forward method:
        d = torch.norm(x_ij)
        epsilon = 1e-7  # Small epsilon to prevent division by zero
        return (-self.A*torch.exp(-d / (self.a + epsilon)) + self.R*torch.exp(-d / (self.r + epsilon)))*(x_ij / (d + epsilon)) #+ V*p_i  # Define assumed kernels
        '''

        # Vectorized calculation of pairwise differences and distances
        x_i = positions.unsqueeze(1) # Shape: (N, 1, D)
        x_j = positions.unsqueeze(0) # Shape: (1, N, D)
        x_ij = x_i - x_j            # Shape: (N, N, D)
        d_ij = torch.norm(x_ij, dim=-1) # Shape: (N, N)

        epsilon = 1e-8

        N = positions.shape[0]

        scalar_part = -A * torch.exp(-d_ij / a) + R * torch.exp(-d_ij / r)    # ***** MODIFY KERNELS HERE !!! ******************************

        scalar_part.diagonal(0).fill_(0.0)  # Avoids unphysical self-interactions

        d_ij_unit_vec = x_ij / (d_ij.unsqueeze(-1) + epsilon) # Normalizing
        d_ij_unit_vec[torch.eye(N, dtype=torch.bool, device=d_ij.device)] = 0.0 # Ensures diagonal vectors are 0, so there's no self-interactions

        known_x_forces_all_pairs = scalar_part.unsqueeze(-1) * d_ij_unit_vec  # multiplies the unit vector to give it the direction
        vx_known = known_x_forces_all_pairs.sum(dim=1) # Sums over j to get the total 'known' force on each particle i, and forms the vector of all of them
                                                  # Shape becomes (N, 2) for the 2-D forces

        return vx_known

We assume no polarity dynamics for now:

In [ ]:
class PolarityKernel(nn.Module):
    def __init__(self):
        super().__init__()
        self.g = nn.Parameter(torch.tensor(1.0)) # g is the parameter for the polarity equation, I'm just using one parameter for now

    def forward(self, x_ij, p_i, p_j):
        return 0  # Assumed polarity kernels

Now we add an Interaction Network module, which consists of two MLPs: one learning object dynamics, and the other learning relational dynamics.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dim=128, depth=2, act=nn.Tanh):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, hidden_dim), act()]
            d = hidden_dim
        layers += [nn.Linear(d, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


    # depth = 2 means that we get two hidden layers: Linear --> Tanh --> Linear --> Tanh --> Linear(out)


class InteractionNet(nn.Module):
    """
    Dense Interaction Network:
      messages m_ij = phi_e(edge_feat_ij)
      aggregate m_i = sum_j m_ij
      output out_i = phi_v([node_feat_i, m_i])

    We will use it to output v_theta (N,2) for positions.
    """
    def __init__(self, node_feat_dim, edge_feat_dim, msg_dim=64, hidden_dim=128, out_dim=2):
        super().__init__()
        self.msg_dim = msg_dim
        self.edge_mlp = MLP(edge_feat_dim, msg_dim, hidden_dim=hidden_dim, depth=2, act=nn.Tanh)
        self.node_mlp = MLP(node_feat_dim + msg_dim, out_dim, hidden_dim=hidden_dim, depth=2, act=nn.Tanh)

    def forward(self, positions, polarities, t_scalar=None):
        """
        positions: (N,2)
        polarities: (N,2)
        t_scalar: scalar tensor (optional) if you want time-dependent interactions
        returns: v_theta (N,2)
        """
        N = positions.shape[0]
        device = positions.device
        dtype = positions.dtype

        # --- node features h_i ---
        # simplest: h_i = [x_i, p_i]  -> (N,4)
        node_feat = torch.cat([positions, polarities], dim=-1)  # (N,4)

        # If you want time dependence, append t as a feature:
        if t_scalar is not None:
            if t_scalar.ndim == 0:
                t_feat = t_scalar.expand(N, 1)          # (N,1)
            else:
                t_feat = t_scalar.reshape(1).expand(N, 1)
            node_feat = torch.cat([t_feat, node_feat], dim=-1)  # (N, 1+4)

        # --- build dense edges (i <- j) ---
        # dx_ij = x_j - x_i
        dx = positions[None, :, :] - positions[:, None, :]      # (N,N,2)
        r2 = (dx ** 2).sum(-1, keepdim=True)                    # (N,N,1)

        dp = polarities[None, :, :] - polarities[:, None, :]    # (N,N,2)

        # edge features: [dx, r2, dp] -> (N,N, 2+1+2=5)
        edge_feat = torch.cat([dx, r2, dp], dim=-1)             # (N,N,5)

        # mask out self edges
        mask = ~torch.eye(N, dtype=torch.bool, device=device)   # (N,N)
        ii, jj = torch.where(mask)                              # (E,), (E,) where E=N*(N-1)

        edge_feat_flat = edge_feat[ii, jj, :]                   # (E, 5)

        # messages per edge
        m_ij = self.edge_mlp(edge_feat_flat)                    # (E, msg_dim)

        # aggregate per receiver i
        m_i = torch.zeros(N, self.msg_dim, device=device, dtype=dtype)
        m_i.index_add_(0, ii, m_ij)                             # sum over incoming edges

        # node update -> v_theta
        node_in = torch.cat([node_feat, m_i], dim=-1)           # (N, node_feat_dim+msg_dim)
        out = self.node_mlp(node_in)                            # (N,2)

        return out

Now we define the right-hand side of the ODEs in Eq (6) using both the Interaction Network model and the known kernels we defined.

NOTE that the following code **does not include polarity dynamics** (yet):

In [ ]:
class ODESystem(nn.Module): # Defines ODE system as a PyTorch module
  def __init__(self, known_kernel_x, global_params, interaction_net, known_kernel_p):
    super().__init__()
    self.known_kernel_x = known_kernel_x
    self.interaction_net = interaction_net
    self.global_params  = global_params  # <-- new
    self.known_kernel_p = known_kernel_p

  def forward(self, t, X):  # Defines the ODE rhs dX/dt = f(t,X), where X = (x1,...xN,p1,...,pN) is the current state of the system

    params = self.global_params()  # dict of scalar tensors

    N = X.shape[0]//2 # Define number of cells here! It's the shape of X divided by 2, since X has both positions and polarities.
    positions = X[:N] # Extracts positions, each entry is a 2D position vector
    polarities = X[N:]

    positions = positions.requires_grad_(True)

    '''---------------------------------------------------------
    THE NEXT PART IS THE KNOWN VELOCITY PART (pre-existing code)
    ------------------------------------------------------------
    '''

    v_known = self.known_kernel_x(positions, params)

    '''----------------------------------------------
    NOW COMES THE LEARNED PART via the InteractionNet
    This returns v_nn with shape (N,2)
    -------------------------------------------------
    '''

    v_nn = self.interaction_net(positions, polarities, t_scalar = t)

    dX = v_known + v_nn

    '''----------------------------------------------------------
    For polarity dynamics, this is zero for this proof of concept
    -------------------------------------------------------------
    '''

    dP = torch.zeros_like(polarities)

    return torch.cat([dX, dP], dim=0) # Concatenates dX and dP


Now we define the **cost functional**. In this implementation, this is the path version of the interaction cost (i.e. the sum of W2 distances along the trajectories in this case).

In [ ]:
from geomloss import SamplesLoss

W2_dist = SamplesLoss(loss="sinkhorn", p=2, blur=0.05)

def cost_function_W2(sim_positions, target_positions, alphaD):
    """
    sim_positions: (steps, N, 2)
    target_positions: (steps, N, 2)
    """

    steps = sim_positions.shape[0]
    N = sim_positions.shape[1]

    # 1) Trajectory matching (W2 per snapshot)
    total_w2 = 0.0
    for k in range(steps):
        total_w2 = total_w2 + W2_dist(sim_positions[k], target_positions[k])


    return alphaD * (total_w2 / steps)

Define the initial conditions, and the target dataset. For this, I upload a CSV file of a simulated dataset.

In [ ]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

Saving (Run=2)_V=0.025_A=0.025_R=0.0375_a=0.25_r=0.125.csv to (Run=2)_V=0.025_A=0.025_R=0.0375_a=0.25_r=0.125.csv


In [ ]:
df = pd.read_csv('(Run=2)_V=0.025_A=0.025_R=0.0375_a=0.25_r=0.125.csv', header=None)

In [ ]:
# We extract the first row of the dataset, i.e. the initial positions and polarities
row0 = df.iloc[0].values  # shape (80,1)

# We split into positions and polarities:
x0_flat = row0[:40]  # first 40 entries
p0_flat = row0[40:]  # next 40 entries

x_init_np = x0_flat.reshape(20, 2)  # shape (20, 2)
p_init_np = p0_flat.reshape(20, 2)  # shape (20, 2)

# Convert them to PyTorch tensors:
x_init = torch.tensor(x_init_np, dtype=torch.float32)  # shape (20, 2)
p_init = torch.tensor(p_init_np, dtype=torch.float32)  # shape (20, 2)

Now we define the "target" dataset as a tensor with shape (timeframes, 2*N, dim) comprised of the 'observed' synthetic data, with all the positions and then all the polarities for each timeframe.

In [ ]:
data_np = df.values  # Convert DataFrame to a NumPy array
# Reshape the data: 200 timeframes, 40 entities (positions + polarities), 2 dimensions (x,y or p_x, p_y)
target_snapshots = torch.tensor(data_np, dtype=torch.float32).reshape(20, 40, 2)
print(target_snapshots.shape)

torch.Size([20, 40, 2])


Now we initialize everything and train:

In [ ]:
from re import X

N = 20
T_final = 95 # Final time for the simulation, must correspond with target dataset
steps = target_snapshots.shape[0] # Number of time steps, should correspond to the one in the target dataset
num_epochs = 100  # Number of training epochs

# Dimensions for the NN models:
input_dim = 3 # This is the input dimension for *each pairwise interaction* (r_ij, p_i)
hidden_dim = 64 # Lower to 32 if it takes too long
output_dim = 2 # Output dimension is 2 for a 2D force

''' NEW: InteractionNet for learned velocity
If we pass t into InteractionNet, node_feat_dim = 5, else 4. '''
use_time_in_interactions = True
node_feat_dim = 5 if use_time_in_interactions else 4
edge_feat_dim = 5        # [dx(2), r2(1), dp(2)] = 5

interaction_net = InteractionNet(
    node_feat_dim=node_feat_dim,
    edge_feat_dim=edge_feat_dim,
    msg_dim=64,
    hidden_dim=128,
    out_dim=2
)


# Initialize known position dynamics
known_x = PosKernels()

# And polarity dynamics
known_p = PolarityKernel()

# And the global parameters module:
kernel_params = GlobalKernelParams()


ode_func = ODESystem(
    known_kernel_x=known_x,
    global_params = kernel_params,
    interaction_net=interaction_net,
    known_kernel_p=known_p
)

# Define parameter groups with different learning rates

LR_NN = 1e-3
LR_known = 1e-3

param_groups = [
    {'params': interaction_net.parameters(), 'lr': LR_NN},
    {'params': known_x.parameters(), 'lr': LR_known},
    {'params': known_p.parameters(), 'lr': LR_known}
]


# Initial condition:
X0 = torch.cat([x_init, p_init], dim=0) # Change to match initial conditions in the dataset, and figure out why "dim=0"

# Time points:
t = torch.linspace(0, T_final, steps)

# Optimizer
optimizer = optim.Adam(param_groups)

In [ ]:
# To run the pipeline on the GPU / CPU:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# These MUST be moved:
X0 = X0.to(device)
target_snapshots = target_snapshots.to(device)
t = t.to(device)
interaction_net = interaction_net.to(device)
ode_func = ode_func.to(device)


Below we initialize the weights / biases of the InteractionNet NN to smaller values. This helps control the initial magnitude of the unknown forces.

In [ ]:
def initialize_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight, gain=0.1) # Using Xavier uniform with a small gain
        if m.bias is not None:
            nn.init.constant_(m.bias, 0) # Initialize biases to zero

interaction_net.apply(initialize_weights)

print("Neural network weights initialized.")

Neural network weights initialized.


Extract the initial parameter values:

In [ ]:
with torch.no_grad():
    p = ode_func.global_params()
    print("A:", float(p["A"]), "a:", float(p["a"]), "R:", float(p["R"]), "r:", float(p["r"]))

A: 0.09531017392873764 a: 0.6931471824645996 R: 0.09531017392873764 r: 0.6931471824645996


The next cell tests how long it takes to integrate the NN-augmented ODE system. Here we re-define the ValueNet neural network explicitly because, for some reason, it was not recognizing it before.

In [ ]:
import time
import torch.nn as nn
start = time.time()

interaction_net = InteractionNet(
    node_feat_dim=node_feat_dim,
    edge_feat_dim=edge_feat_dim,
    msg_dim=64,
    hidden_dim=128,
    out_dim=2
)
ode_func = ODESystem(known_kernel_x = known_x, global_params = kernel_params, interaction_net = interaction_net, known_kernel_p = known_p)

# Apply weight initialization and move to device again for the new instances
interaction_net.apply(initialize_weights)

interaction_net = interaction_net.to(device)
ode_func = ode_func.to(device)

''' If I encounter this error again:
OutOfMemoryError: CUDA out of memory indicates that the GPU ran out of memory during the ODE integration.
This is likely due to the memory required for the vectorized ODESystem and the ODE solver, which scales with N*N.
To fix this, we try reducing the hidden dimension of the neural networks (hidden_dim) in the initialization cell above '''

X_pred = odeint(ode_func, X0, t, method='dopri5', rtol=1e-3, atol=1e-4)

print("Integration time:", time.time() - start)


Integration time: 0.11731767654418945


Below is a **protoype** of the new training loop. Use the following cell to **test changes only**.

In [ ]:
# Test how long it takes to run one epoch of training:

import time
start = time.time()

# Define dim, alpha1, alpha2
dim = 2 # Dimension of position/polarity vectors
alphaD = 1

optimizer = torch.optim.AdamW(ode_func.parameters(), lr=1e-3, weight_decay=1e-6)


optimizer.zero_grad()
X_pred = odeint(ode_func, X0, t, method='dopri5', rtol=1e-3, atol=1e-4)
Pred_pos = X_pred[:, :N] # Corrected slicing for positions over all time steps
Pred_pol = X_pred[:, N:] # Corrected slicing for polarities over all time steps


''' Now we calculate the loss (a.k.a. cost) '''
# Pass dX_traj and dP_traj lists to the cost function
loss = cost_function_W2(X_pred, target_snapshots, alphaD)
loss.backward()
optimizer.step()

print("One epoch of training:", time.time() - start , f"Loss: {loss.item()}")


One epoch of training: 0.9286074638366699 Loss: 0.4913644790649414


Here now we do the full training, training for a total of "num_epochs" epochs.

**NOTE:** Very good results were obtained in the IntNet setting by setting LR_known to be bigger than LR_NN (around 1 order of magnitude bigger usually)

In [ ]:
num_epochs = 500

alphaD = 5e3

LR_NN = 1e-7        # Start with 1e-4 and go from there
LR_known = 1e-7     # Start with 1e-4 and go from there

wd = 1e-9            # weight decay parameter, keep one to two orders of maginuted smaller than the LR's


rtol = 1e-3     # Tolerances for the integrator
atol = 1e-4

# Automate the training loop for num_epochs times
import time

optimizer = torch.optim.AdamW(
    [{'params': interaction_net.parameters(), 'lr': LR_NN, 'weight_decay': wd},
     {'params': ode_func.global_params.parameters(), 'lr': LR_known, 'weight_decay': 0.0},]
)   # weight decay for the known parameters is usually zero


torch.nn.utils.clip_grad_norm_(interaction_net.parameters(), 1.0)   # gradient clipping, to avoid overflows
torch.nn.utils.clip_grad_norm_(kernel_params.parameters(), 1.0)


ode_func = ODESystem(known_kernel_x=known_x, global_params = kernel_params, interaction_net = interaction_net, known_kernel_p=known_p)
ode_func.to(device)


# Re-define W2_dist for self-containment within this cell
W2_dist = SamplesLoss(loss="sinkhorn", p=2, blur=0.05)

total_training_start_time = time.time()

# num_epochs, N, dim, alpha1, alpha2, optimizer, ode_func, X0, t,
# value_net, Hamiltonian, Target_pos are assumed to be defined globally

print(f"Starting training for {num_epochs} epochs...")
for epoch in range(num_epochs):
  epoch_start_time = time.time() # Start timer for this epoch

  optimizer.zero_grad()
  X_pred = odeint(ode_func, X0, t, method='dopri5', rtol=rtol, atol=atol)
  Pred_pos = X_pred[:, :N]
  Pred_pol = X_pred[:, N:]


  # Calculate the loss (a.k.a. cost)
  loss = cost_function_W2(X_pred, target_snapshots, alphaD)

  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()


  print(f"Epoch {epoch+1}/{num_epochs} | Time: {time.time() - epoch_start_time:.4f}s | Loss: {loss.item():.4f}")

total_training_end_time = time.time()
print(f"Total training for {num_epochs} epochs completed in {total_training_end_time - total_training_start_time:.4f}s")

Starting training for 500 epochs...
Epoch 1/500 | Time: 0.8008s | Loss: 16.1895
Epoch 2/500 | Time: 0.7468s | Loss: 16.1899
Epoch 3/500 | Time: 0.7407s | Loss: 16.1908
Epoch 4/500 | Time: 0.7666s | Loss: 16.1908
Epoch 5/500 | Time: 0.7659s | Loss: 16.1913
Epoch 6/500 | Time: 0.7267s | Loss: 16.1912
Epoch 7/500 | Time: 0.7436s | Loss: 16.1918
Epoch 8/500 | Time: 0.9770s | Loss: 16.1925
Epoch 9/500 | Time: 1.0623s | Loss: 16.1922
Epoch 10/500 | Time: 1.1387s | Loss: 16.1921
Epoch 11/500 | Time: 0.8122s | Loss: 16.1917
Epoch 12/500 | Time: 0.7475s | Loss: 16.1919
Epoch 13/500 | Time: 0.7263s | Loss: 16.1913
Epoch 14/500 | Time: 0.7672s | Loss: 16.1910
Epoch 15/500 | Time: 0.8064s | Loss: 16.1908
Epoch 16/500 | Time: 0.6454s | Loss: 16.1907
Epoch 17/500 | Time: 0.7114s | Loss: 16.1903
Epoch 18/500 | Time: 0.7974s | Loss: 16.1893
Epoch 19/500 | Time: 0.7624s | Loss: 16.1896
Epoch 20/500 | Time: 0.7528s | Loss: 16.1895
Epoch 21/500 | Time: 0.7939s | Loss: 16.1898
Epoch 22/500 | Time: 1.1418s

Below is a different version of the training loop that also plots the loss across all training epochs:

In [ ]:
import matplotlib.pyplot as plt
import time

num_epochs = 1500

alphaD = 1e4

LR_NN = 1e-7
LR_known = 1e-7

wd = 1e-8

rtol = 1e-3
atol = 1e-4

# --- Optimizer Definition ---
optimizer = optim.AdamW(
    [{'params': interaction_net.parameters(), 'lr': LR_NN, 'weight_decay': wd},
     {'params': kernel_params.parameters(), 'lr': LR_known, 'weight_decay': 0.0}]
)

# --- ODESystem Re-initialization (to ensure correct state after edits/runs) ---
ode_func = ODESystem(known_kernel_x=known_x, global_params=kernel_params, interaction_net=interaction_net, known_kernel_p=known_p)
ode_func.to(device)

# --- Re-define W2_dist for self-containment ---
W2_dist = SamplesLoss(loss="sinkhorn", p=2, blur=0.05)

# --- List to store loss values for plotting ---
loss_history = []

total_training_start_time = time.time()

print(f"Starting training for {num_epochs} epochs...")
for epoch in range(num_epochs):
  epoch_start_time = time.time()

  optimizer.zero_grad() # Clear gradients from previous step

  # --- ODE Integration ---
  X_pred = odeint(ode_func, X0, t, method='dopri5', rtol=rtol, atol=atol)
  Pred_pos = X_pred[:, :N]
  Pred_pol = X_pred[:, N:]

  # --- Loss Calculation ---
  loss = cost_function_W2(X_pred, target_snapshots, alphaD)
  loss_history.append(loss.item()) # Store the loss

  # --- Backpropagation and Optimization ---
  optimizer.zero_grad(set_to_none=True) # A more efficient way to clear gradients
  loss.backward() # Backpropagation

  # Gradient clipping (corrected placement to be after loss.backward() and before optimizer.step())
  torch.nn.utils.clip_grad_norm_(interaction_net.parameters(), 1.0)
  torch.nn.utils.clip_grad_norm_(kernel_params.parameters(), 1.0)

  optimizer.step() # Update model parameters

  print(f"Epoch {epoch+1}/{num_epochs} | Time: {time.time() - epoch_start_time:.4f}s | Loss: {loss.item():.4f}")

total_training_end_time = time.time()
print(f"Total training for {num_epochs} epochs completed in {total_training_end_time - total_training_start_time:.4f}s")

# --- Plotting the loss history ---
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), loss_history, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Verify that the models are still set to training mode

print(ode_func.training)
print(ode_func.interaction_net.training)

NameError: name 'ode_func' is not defined

Extract learned parameters:

In [ ]:
with torch.no_grad():
    print(ode_func.global_params())

NameError: name 'torch' is not defined

Save the trained NN models:

In [ ]:
# Save the trained NN models
torch.save(known_x.state_dict(), 'known_x_model.pth')
torch.save(interaction_net.state_dict(), 'interaction_net_model.pth')
torch.save(known_p.state_dict(), 'known_p_model.pth')
torch.save(global_params.state_dict(), 'global_params.pth')

print("Trained NN models saved.")

In [ ]:
files.download('known_x_model.pth')
files.download('interaction_net_model.pth')
files.download('known_p_model.pth')
files.download('global_params.pth')

Animation code


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np # Import numpy for min/max calculations

# N is defined from previous cells: N = 20
# Extract positions from X_pred, the tensor from the last odeint call (from training or simulation) with shape (timeframes, 2*N, dim)
# The first N entries are positions, next N are polarities.
positions_pred = X_pred[:, :N, :] # Shape (timeframes, N, dim)

# Extract positions from target_snapshots, which also has shape (timeframes, 2*N, dim)
target_positions = target_snapshots[:, :N, :] # Shape (timeframes, N, dim)

# Convert positions to numpy for plotting
positions_np = positions_pred.cpu().detach().numpy()
target_positions_np = target_positions.cpu().detach().numpy()

# Determine overall min/max for plot limits, considering both predicted and target data
all_x_coords = np.concatenate((positions_np[:, :, 0].flatten(), target_positions_np[:, :, 0].flatten()))
all_y_coords = np.concatenate((positions_np[:, :, 1].flatten(), target_positions_np[:, :, 1].flatten()))

min_x, max_x = all_x_coords.min() - 0.5, all_x_coords.max() + 0.5
min_y, max_y = all_y_coords.min() - 0.5, all_y_coords.max() + 0.5


# Set up the figure and axes
fig, ax = plt.subplots(figsize=(10, 10)) # Increased figure size for better visibility
ax.set_xlim(min_x, max_x)
ax.set_ylim(min_y, max_y)
ax.set_title('Predicted vs. Target Cell Positions Over Time')
ax.set_xlabel('X-coordinate')
ax.set_ylabel('Y-coordinate')
ax.set_aspect('equal', adjustable='box')
plt.grid(True)

# Initialize the plot with the first frame's data
line_pred, = ax.plot([], [], 'o', markersize=5, color='blue', label='Predicted')
line_target, = ax.plot([], [], 'o', markersize=5, color='red', label='Target')
ax.legend()

def init():
    line_pred.set_data([], [])
    line_target.set_data([], [])
    return line_pred, line_target, # Return all artists that will be updated

def animate(i):
    # Update the data for each frame for predicted trajectory
    x_coords_pred = positions_np[i, :, 0]
    y_coords_pred = positions_np[i, :, 1]
    line_pred.set_data(x_coords_pred, y_coords_pred)

    # Update the data for each frame for target trajectory
    x_coords_target = target_positions_np[i, :, 0]
    y_coords_target = target_positions_np[i, :, 1]
    line_target.set_data(x_coords_target, y_coords_target)

    return line_pred, line_target,

# Create the animation
# frames: number of time steps (length of the first dimension of positions_np)
# interval: delay between frames in ms
# blit=True means only re-draw the parts that have changed.
ani = animation.FuncAnimation(fig, animate, frames=min(positions_np.shape[0], target_positions_np.shape[0]), interval=100, blit=True)

# Save the animation as a GIF
gif_path = 'predicted_vs_target_cell_movement.gif'
ani.save(gif_path, writer='pillow', fps=2) # fps controls the speed of the animation

print(f"Animation saved to {gif_path}")

# Display the animation in the notebook (optional)
plt.close(fig) # Close the static plot window
HTML(ani.to_jshtml())

Animation saved to predicted_vs_target_cell_movement.gif


Download animation as gif:

In [ ]:
from google.colab import files
files.download("predicted_vs_target_cell_movement.gif")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Code for extracting the v_NN vectors from the final learned models:

In [ ]:
def save_vnn_along_traj_csv(
    ode_func,
    odeint_fn,           # torchdiffeq.odeint or odeint_adjoint
    X0,                  # (2N,2) state at t0
    t_eval,              # (steps,) time grid tensor
    csv_path="v_nn.csv",
    method="dopri5",
    rtol=1e-6,
    atol=1e-8,
):
    """
    Saves v_nn(t_k) for each timestep to a CSV.

    Output shape: (steps, N, 2)
    CSV storage: (steps, 2*N) columns [cell0_vx, cell0_vy, cell1_vx, ...]
    """
    ode_func.eval()

    # 1) Integrate to get the trajectory X(t_k)
    with torch.no_grad():
        X_traj = odeint_fn(ode_func, X0, t_eval, method=method, rtol=rtol, atol=atol)
        # X_traj shape has shape (steps, 2N, 2) if X0 is (2N,2)

    steps = X_traj.shape[0]
    twoN  = X_traj.shape[1]
    N     = twoN // 2

    # 2) Compute v_nn at each saved timestep
    vnn_list = []
    with torch.no_grad():
        for k in range(steps):
            t_k = t_eval[k]
            X_k = X_traj[k]                 # (2N,2)
            positions  = X_k[:N]            # (N,2)
            polarities = X_k[N:]            # (N,2)

            v_nn = ode_func.interaction_net(positions, polarities, t_scalar=t_k)  # (N,2)
            vnn_list.append(v_nn)

    vnn = torch.stack(vnn_list, dim=0)      # (steps, N, 2)

    # 3) Write CSV: flatten (N,2) -> (2N) per timestep
    vnn_flat = vnn.reshape(steps, 2 * N).cpu().numpy()

    cols = []
    for i in range(N):
        cols += [f"cell{i}_vx", f"cell{i}_vy"]

    df = pd.DataFrame(vnn_flat, columns=cols)
    df.insert(0, "t", t_eval.detach().cpu().numpy())

    df.to_csv(csv_path, index=False)

    return vnn  # returns tensor of shape (steps, N, 2) for immediate analysis

In [ ]:
vnn = save_vnn_along_traj_csv(
    ode_func=ode_func,
    odeint_fn=odeint,
    X0=X0,
    t_eval=t,
    csv_path="v_nn.csv",
)
print(vnn.shape)  # (steps, N, 2)

torch.Size([20, 20, 2])


In [ ]:
files.download('v_nn.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Analysis of v_NN vectors**

In [ ]:
from google.colab import files

files.upload()

Saving v_nn_0224.csv to v_nn_0224.csv


{'v_nn_0224.csv': b't,cell0_vx,cell0_vy,cell1_vx,cell1_vy,cell2_vx,cell2_vy,cell3_vx,cell3_vy,cell4_vx,cell4_vy,cell5_vx,cell5_vy,cell6_vx,cell6_vy,cell7_vx,cell7_vy,cell8_vx,cell8_vy,cell9_vx,cell9_vy,cell10_vx,cell10_vy,cell11_vx,cell11_vy,cell12_vx,cell12_vy,cell13_vx,cell13_vy,cell14_vx,cell14_vy,cell15_vx,cell15_vy,cell16_vx,cell16_vy,cell17_vx,cell17_vy,cell18_vx,cell18_vy,cell19_vx,cell19_vy\n0.0,0.03030067,-0.00084830465,0.029005127,-0.0077942,0.022538457,0.0040071765,0.026789598,-0.0044964952,0.025611999,-0.003914019,0.021177365,-0.0069876444,0.035404667,-0.0014805084,0.040161252,-0.00061570125,0.023770934,0.0031220699,0.017902175,-0.0008812576,0.033001512,0.005841485,0.020987736,-0.0074768714,0.02721546,-0.0027066702,0.01967716,-0.0009463489,0.024019597,0.000573074,0.018992081,0.0016549509,0.03193949,-0.0044676904,0.0312087,0.0056434027,0.042294152,-0.0062517854,0.024509463,-0.00052201544\n5.263158,0.029323729,-0.0005920521,0.027828773,-0.00663587,0.021826373,0.004084005,0.02

In [ ]:
import pandas as pd

nn_contribs_df = pd.read_csv('v_nn_0224.csv')

In [ ]:
N = 20
num_timeframes = 19 # Total number of timeframes to analyze

In [ ]:
nn_contribs_df

,t,cell0_vx,cell0_vy,cell1_vx,cell1_vy,cell2_vx,cell2_vy,cell3_vx,cell3_vy,cell4_vx,...,cell15_vx,cell15_vy,cell16_vx,cell16_vy,cell17_vx,cell17_vy,cell18_vx,cell18_vy,cell19_vx,cell19_vy
0,0.000000,0.030301,-8.483047e-04,0.029005,-0.007794,0.022538,0.004007,0.026790,-0.004496,0.025612,...,0.018992,0.001655,0.031939,-0.004468,0.031209,0.005643,0.042294,-0.006252,0.024509,-0.000522
1,5.263158,0.029324,-5.920521e-04,0.027829,-0.006636,0.021826,0.004084,0.026305,-0.003774,0.025374,...,0.019159,0.001782,0.030541,-0.003708,0.029567,0.005563,0.038792,-0.005100,0.024562,-0.000358
2,10.526316,0.028417,-3.627362e-04,0.026847,-0.005544,0.021319,0.004077,0.025870,-0.003092,0.025153,...,0.019489,0.001834,0.029295,-0.003003,0.028150,0.005389,0.035794,-0.004081,0.024580,-0.000205
3,15.789474,0.027606,-1.646308e-04,0.026064,-0.004536,0.021020,0.003987,0.025501,-0.002463,0.024962,...,0.019951,0.001819,0.028220,-0.002364,0.026974,0.005128,0.033275,-0.003191,0.024578,-0.000067
4,21.052631,0.026913,-6.186310e-07,0.025474,-0.003625,0.020926,0.003818,0.025208,-0.001898,0.024816,...,0.020509,0.001747,0.027326,-0.001799,0.026042,0.004790,0.031205,-0.002429,0.024573,0.000053
5,26.315788,0.026350,1.278403e-04,0.025065,-0.002821,0.021024,0.003576,0.024999,-0.001406,0.024722,...,0.021127,0.001628,0.026615,-0.001313,0.025351,0.004387,0.029548,-0.001790,0.024579,0.000151
6,31.578947,0.025920,2.206452e-04,0.024819,-0.002130,0.021292,0.003272,0.024876,-0.000989,0.024687,...,0.021772,0.001476,0.026080,-0.000909,0.024887,0.003935,0.028263,-0.001266,0.024610,0.000225
7,36.842106,0.025619,2.788654e-04,0.024713,-0.001552,0.021698,0.002919,0.024833,-0.000651,0.024710,...,0.022416,0.001302,0.025708,-0.000584,0.024626,0.003450,0.027304,-0.000850,0.024674,0.000273
8,42.105263,0.025437,3.045211e-04,0.024720,-0.001087,0.022206,0.002533,0.024862,-0.000388,0.024787,...,0.023034,0.001115,0.025480,-0.000337,0.024537,0.002949,0.026623,-0.000533,0.024774,0.000294
9,47.368420,0.025356,3.002756e-04,0.024814,-0.000726,0.022775,0.002131,0.024952,-0.000197,0.024909,...,0.023610,0.000925,0.025371,-0.000160,0.024584,0.002452,0.026172,-0.000303,0.024906,0.000290


Now we reintroduce the code to save the predicted positions CSV file:

In [ ]:
from tqdm.auto import tqdm

def simulate_system(num_timeframes, time_step_size, initial_conditions, known_kernel_x, global_params_model, interaction_net, known_kernel_p):

  N = initial_conditions.shape[0] // 2
  dim = 2 # Assuming 2D dimensions for positions and polarities

  # Instantiate the ODE system with the provided models and parameters:
  ode_func = ODESystem(known_kernel_x=known_kernel_x, global_params=global_params_model, interaction_net=interaction_net, known_kernel_p=known_kernel_p)
  device = initial_conditions.device
  ode_func.to(device)

  X0 = initial_conditions.to(device)

  # Define time points for simulation. num_timeframes here refers to the total number of snapshots including t=0.
  # If num_timeframes = 20 (like in the loaded target_snapshots), then t_eval runs from 0 to (19 * time_step_size).
  t_eval = torch.linspace(0, (num_timeframes - 1) * time_step_size, num_timeframes).to(device)

  # Perform ODE integration for the entire trajectory
  print(f"Simulating system for {num_timeframes} snapshots (time from {t_eval[0]:.2f} to {t_eval[-1]:.2f})...")
  with torch.no_grad():
    X_traj = odeint(ode_func, X0, t_eval, method='dopri5', rtol=1e-3, atol=1e-4) # Using same tolerances as training
  print("Simulation complete.")

  # Create empty lists to store the processed data for CSVs
  predicted_positions_list = [] # Will store (2N*dim) flattened vector for each snapshot (positions and polarities)
  known_x_vectors_list = []     # Will store (N*dim) flattened vector for known_x for each snapshot
  known_p_vectors_list = []     # Will store (N*dim) flattened vector for known_p for each snapshot
  vnn_x_vectors_list = []       # Will store (N*dim) flattened vector for vnn_x for each snapshot

  # Process each snapshot in the trajectory
  for k in tqdm(range(num_timeframes), desc="Processing simulation snapshots for component vectors"):
      t_k = t_eval[k]
      X_k = X_traj[k]               # X_k shape is (2N, 2)
      positions_k = X_k[:N]         # positions_k shape is (N, 2)
      polarities_k = X_k[N:]        # polarities_k shape is (N, 2)

      # Store predicted positions (flattened x and p together)
      predicted_positions_list.append(X_k.cpu().numpy().flatten()) # Shape (2N*dim)

      # Calculate and store known_x vectors
      # global_params_model is the GlobalKernelParams instance, so call it to get the dict of params
      current_known_x = known_kernel_x(positions_k, global_params_model()).cpu().numpy().flatten() # Shape (N*dim)
      known_x_vectors_list.append(current_known_x)

      # Calculate and store known_p vectors (currently all zeros as per PolarityKernel definition, for the proof of concept)
      # The PolarityKernel.forward method currently returns 0. So, we'll append zeros for known_p.
      known_p_vectors_list.append(torch.zeros_like(polarities_k).cpu().numpy().flatten()) # Shape (N*dim)

      # Calculate v_nn (only x part, as dP from ODE system is hardcoded to 0)
      current_vnn_x = interaction_net(positions_k, polarities_k, t_scalar=t_k).cpu().numpy().flatten() # Shape (N*dim)
      vnn_x_vectors_list.append(current_vnn_x)

  # Convert lists of flattened vectors to NumPy arrays
  predicted_positions_np = np.array(predicted_positions_list)
  known_x_vectors_np = np.array(known_x_vectors_list)
  known_p_vectors_np = np.array(known_p_vectors_list)
  vnn_x_vectors_np = np.array(vnn_x_vectors_list)

  # Define column names for each CSV file
  predicted_cols = ["t"]
  for i in range(1, N + 1):
      predicted_cols.extend([f'x_{i}_dim1', f'x_{i}_dim2'])
  for i in range(1, N + 1):
      predicted_cols.extend([f'p_{i}_dim1', f'p_{i}_dim2'])

  known_x_cols = ["t"]
  for i in range(1, N + 1):
      known_x_cols.extend([f'known_x_{i}_dim1', f'known_x_{i}_dim2'])

  known_p_cols = ["t"]
  for i in range(1, N + 1):
      known_p_cols.extend([f'known_p_{i}_dim1', f'known_p_{i}_dim2'])

  vnn_x_cols = ["t"]
  for i in range(1, N + 1):
      vnn_x_cols.extend([f'vnn_x_{i}_dim1', f'vnn_x_{i}_dim2'])

  # Add time column to the data before creating DataFrames
  t_eval_np = t_eval.cpu().numpy().reshape(-1, 1)

  predicted_df = pd.DataFrame(np.hstack((t_eval_np, predicted_positions_np)), columns=predicted_cols)
  known_x_df = pd.DataFrame(np.hstack((t_eval_np, known_x_vectors_np)), columns=known_x_cols)
  known_p_df = pd.DataFrame(np.hstack((t_eval_np, known_p_vectors_np)), columns=known_p_cols)
  vnn_x_df = pd.DataFrame(np.hstack((t_eval_np, vnn_x_vectors_np)), columns=vnn_x_cols) # Include vnn_x for completeness

  # Save to CSV files
  predicted_df.to_csv("predicted_trajectory.csv", index=False)
  known_x_df.to_csv("known_x_vectors.csv", index=False)
  known_p_df.to_csv("known_p_vectors.csv", index=False)
  vnn_x_df.to_csv("vnn_x_vectors.csv", index=False) # Saving this as well for analysis

  print("Predicted trajectory saved to predicted_trajectory.csv")
  print("Known X vectors saved to known_x_vectors.csv")
  print("Known P vectors saved to known_p_vectors.csv")
  print("VNN X vectors saved to vnn_x_vectors.csv")

  return predicted_df, known_x_df, known_p_df, vnn_x_df


In [ ]:
# Call the simulate_system function using the global variables already defined
# (N, T_final, steps, X0, known_x, kernel_params, interaction_net, known_p)

# Adjust num_timeframes and time_step_size to match existing data/simulation setup
# 'steps' from the data loading determines number of snapshots (e.g., 20)
# 'T_final' determines the total time (e.g., 95 in this case)

actual_num_snapshots = steps # From the data loading, target_snapshots.shape[0]
calculated_time_step_size = T_final / (actual_num_snapshots - 1) if actual_num_snapshots > 1 else T_final

# Call the function with global parameters
predicted_traj_df, known_x_df_out, known_p_df_out, vnn_x_df_out = simulate_system(
    num_timeframes=actual_num_snapshots, # e.g., 20
    time_step_size=calculated_time_step_size, # e.g., T_final / 19
    initial_conditions=X0, # initial state (positions and polarities)
    known_kernel_x=known_x,
    global_params_model=kernel_params, # the GlobalKernelParams instance
    interaction_net=interaction_net,
    known_kernel_p=known_p
)

# Optional: Display the head of one of the generated dataframes
print("\nHead of predicted_trajectory.csv:")
print(predicted_traj_df.head())

from google.colab import files
files.download("predicted_trajectory.csv")
files.download("known_x_vectors.csv")
files.download("known_p_vectors.csv")
files.download("vnn_x_vectors.csv")